# 03 · Filter & Rank — the shared multi-layer antibody filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 14** we run the **antibody** cutoffs on the VHH pool and report honest survival (D3 pt 1).

Run `00`–`02` first so `results/designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

In [ ]:
import filtering_pipeline as fp
import pandas as pd
print("antibody cutoffs:", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

Map each VHH onto the shared `Design` record (`design_type="antibody"`). CDR geometry + developability
ride along in `extra`. Layer 2 (orthogonal) needs a *second* predictor (IgFold/ESMFold) — we run
layers **(1, 3)** here and note L2 is added with a real second predictor.

In [ ]:
df = pd.read_csv("results/designs.csv")
designs = [fp.Design(design_id=str(r.design_id), sequence=str(r.sequence), design_type="antibody",
                     plddt=r.plddt, pae_interaction=r.pae_interaction, scrmsd=r.scrmsd,
                     extra={"cdr_geom": r.cdr_geom, "tap_score": r.tap_score,
                            "humanness": r.humanness, "camsol_like": r.camsol_like, "synthetic": True})
           for r in df.itertuples()]
print(len(designs), "Design objects built (design_type='antibody')")

## Run the pipeline + report

In [ ]:
ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(ranked, top_n=10, save_prefix="results/proj14")
print("\nNOTE: numbers are SYNTHETIC (mock). Layer 2 (orthogonal IgFold/ESMFold) is added in a real run.")
top

## Survival-at-each-layer (honest accounting)
Report N pass / N generated at each layer. De novo antibody hit rates are LOW — survivors are screening inputs.

In [ ]:
if "layers_passed" in ranked:
    print(ranked["layers_passed"].value_counts().sort_index())

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type='antibody'`).
- [ ] Survival-at-each-layer reported (honest, low hit rate).
- [ ] Mapping assumptions written down.

**Next:** `04_validate.ipynb` — developability gate + cross-strain breadth.